<h3 style="color:#6FA8DC; font-weight:bold">01 — Complete Case Analysis (CCA) / Removing Records</h3>

Handling Missing Data — Feature Engineering

This notebook follows the structure and examples of the provided CampusX Day 35 reference notebook, then adds the modern ML workflow and production considerations.

<h5 style="color:#78B89A; font-weight:bold;">1. What is missing data? → simple meaning</h5>

Missing data means some values in a dataset are not available. In Pandas they are commonly represented by `NaN`.

Example: if a person's `experience` is not recorded, that feature contains a missing value.

Before choosing a technique, first measure **how much data is missing and where**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('data_science_job.csv')
df.head()

In [ ]:
df.isnull().mean().sort_values(ascending=False) * 100

<h5 style="color:#78B89A; font-weight:bold;">2. What is Complete Case Analysis? → remove incomplete rows</h5>

**CCA means:** keep only rows where the selected features contain no missing values.

```text
Original data
     ↓
Find rows with missing values
     ↓
Remove those rows
     ↓
Complete cases only
```

In Pandas, the basic operation is `dropna()`. In the reference notebook, CCA is applied to columns with a relatively small percentage of missing values.

In [ ]:
cols = [var for var in df.columns if df[var].isnull().mean() < 0.05 and df[var].isnull().mean() > 0]
cols

In [ ]:
new_df = df[cols].dropna()
print('Original shape:', df.shape)
print('After CCA:', new_df.shape)
print('Rows removed:', len(df) - len(new_df))
print('Percentage of rows retained:', round(len(new_df) / len(df) * 100, 2), '%')

<h5 style="color:#78B89A; font-weight:bold;">3. Why not simply remove every row? → missingness percentage matters</h5>

If a dataset has only a few missing rows, removing them may be acceptable. If a feature has many missing values, CCA can throw away a large amount of useful data.

A useful first check is:

`missing percentage = missing values / total rows × 100`

The reference notebook specifically filters columns having **less than 5% missingness** before demonstrating CCA.

<h5 style="color:#78B89A; font-weight:bold;">4. When should CCA be used? → practical conditions</h5>

CCA is more reasonable when:

- Missing values are relatively small in number.
- Removing the affected observations does not remove an important population group.
- The remaining data is still representative.
- The application can tolerate losing some rows.
- The missingness is plausibly unrelated to the target/outcome after considering the available information.

⚠️ The last point matters: if missingness is systematically related to the target or to a subgroup, deleting rows can introduce bias.

<h5 style="color:#78B89A; font-weight:bold;">5. Advantages → why CCA is attractive</h5>

- Very simple to understand.
- Very easy to implement.
- No artificial values are inserted.
- Preserves the original values of retained observations.
- No imputation model is required.
- Easy to reproduce in a pipeline when the rule is fixed.

<h5 style="color:#78B89A; font-weight:bold;">6. Disadvantages → what can go wrong</h5>

- You lose observations.
- Effective sample size becomes smaller.
- If missingness is not random, the retained sample can become biased.
- Important minority cases may disappear.
- With several columns, the number of complete rows can fall quickly.
- A method that works on a small percentage of missing values may become unusable as missingness grows.

<h5 style="color:#78B89A; font-weight:bold;">7. Effect on production → why this matters</h5>

In production, new records may naturally contain missing values.

If your training process used CCA but your production system receives a row with a missing feature, you have to decide what the production pipeline should do.

```text
Training:
missing row → removed

Production:
new row + missing value
        ↓
cannot simply 'remove the customer' without considering the application
```

So CCA is mainly a **training-data cleaning strategy**. For a production prediction pipeline, explicit imputation or another missing-value strategy is often needed so the model can accept incomplete records.

<h5 style="color:#78B89A; font-weight:bold;">8. Effect of CCA on distributions → reference notebook check</h5>

The Day 35 notebook compares the distributions before and after CCA. The idea is important: if the distribution changes substantially, deleting incomplete rows has changed the population being modeled.

In [ ]:
if 'training_hours' in df.columns and 'training_hours' in new_df.columns:
    fig, ax = plt.subplots(figsize=(7,4))
    df['training_hours'].hist(bins=50, density=True, ax=ax, alpha=0.5, label='Original')
    new_df['training_hours'].hist(bins=50, density=True, ax=ax, alpha=0.5, label='After CCA')
    ax.legend(); ax.set_title('Training Hours: Before vs After CCA'); plt.show()

<h5 style="color:#78B89A; font-weight:bold;">9. Modern ML way → split first, then learn preprocessing</h5>

For supervised ML, avoid using the full dataset to make preprocessing decisions before the train/test split.

```text
Raw data
   ↓
Train / Test split
   ↓
Learn preprocessing only from training data
   ↓
Apply the same rule to test / production data
```

For CCA specifically, a common modern pattern is to decide which features are eligible for CCA using the training data and then apply the same row-validity rule consistently. For more general ML pipelines, imputation transformers are usually easier to integrate with `Pipeline` and cross-validation.

In [ ]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Example: inspect missingness using ONLY the training set
missing_train = X_train.isnull().mean().sort_values(ascending=False) * 100
missing_train.head(10)

<h5 style="color:#78B89A; font-weight:bold;">10. CCA with a selected feature set → safer pattern</h5>

If you deliberately choose a small set of low-missingness features for CCA, the row filter can be applied to those features.

```python
train_complete = X_train.dropna(subset=selected_columns)
```

This is different from blindly doing `X_train.dropna()` on every column. The latter can delete rows because of an irrelevant feature you may not even use.

In [ ]:
selected_columns = [c for c in X_train.columns if X_train[c].isnull().mean() < 0.05]
train_complete = X_train.dropna(subset=selected_columns)
print('Selected columns:', selected_columns)
print('Before:', X_train.shape)
print('After CCA on selected columns:', train_complete.shape)

<h5 style="color:#78B89A; font-weight:bold;">11. CCA vs Imputation → quick comparison</h5>

| Approach | What happens? | Main benefit | Main risk |
|---|---|---|---|
| CCA | Remove incomplete rows | Very simple | Data loss / bias |
| Mean/Median | Replace missing numerical values | Keeps rows | Can distort distribution |
| Arbitrary value | Replace with chosen constant | Explicit missing code | Can create extreme values |
| Random sample | Replace using observed values | Can preserve distribution better | Randomness / implementation complexity |
| Missing indicator | Add a missingness flag | Preserves information about missingness | Adds features |

The next notebooks cover the numerical methods in detail.

<h5 style="color:#78B89A; font-weight:bold;">12. Final revision → remember this</h5>

```text
Missing data
    ↓
Measure missingness
    ↓
Small + acceptable loss?
    ↓
CCA can be considered
    ↓
Remove incomplete records
    ↓
Check whether distributions / population changed
    ↓
For production ML → use a consistent preprocessing pipeline
```

**One-line definition:** Complete Case Analysis is a missing-data technique in which observations containing missing values in the selected variables are removed.